# PKH Context Sniper — Colab launcher

Запуск ингеста и поиска по архиву чатов Google AI Studio.

**Перед запуском:** `Runtime → Change runtime type → T4 GPU`. На CPU ингест больших архивов идёт часами.

## 1. Монтирование Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Получение кода

Замени URL на свой репозиторий (или закинь файлы в `/content/PKH` вручную).

In [ ]:
import os, sys
REPO_URL = 'https://github.com/LilSheva/PKH.git'
REPO_DIR = '/content/PKH'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 3. Зависимости

Первый запуск — ~3-5 минут (torch уже стоит, ставится только chromadb + sentence-transformers).

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

## 4. Проверка GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 5. Пути и выбор модели

Подправь `CHATS_DIR` под свою папку с дампами на Drive. `DB_DIR` тоже на Drive — чтобы база переживала рестарты Colab.

In [ ]:
from pathlib import Path
import config

config.CHATS_DIR = Path('/content/drive/MyDrive/ai_studio_dumps')
config.DB_DIR = Path('/content/drive/MyDrive/pkh_chroma')

# Доступные модели: 'bge-m3' | 'qwen3' | 'e5-instruct'
config.EMBEDDING_MODEL = 'bge-m3'

# Размышления модели в дампах: 'OFF' | 'ON' | 'SMART'
config.THOUGHT_MODE = 'SMART'

print('chats:', config.CHATS_DIR)
print('db:   ', config.DB_DIR)
print('model:', config.EMBEDDING_MODEL)

## 6. Ингест

Идемпотентно — повторные запуски обрабатывают только новые/изменённые файлы.

In [ ]:
from sniper import ContextSniper

sniper = ContextSniper(embedding_model=config.EMBEDDING_MODEL)
stats = sniper.ingest(config.CHATS_DIR)
print(stats)
print(sniper.stats())

## 7. Поиск контекста

In [ ]:
query = 'как я настраивал ChromaDB на Colab'
res = sniper.search_context(query, top_k=5)
for i, (doc, meta, dist) in enumerate(zip(
    res['documents'][0], res['metadatas'][0], res['distances'][0]
)):
    print(f'--- #{i+1} | chat={meta["chat_id"]} | dist={dist:.3f} ---')
    print(doc[:500])
    print()

## 8. Супер-промпт

In [ ]:
prompt = sniper.generate_super_prompt(
    main_prompt='Напиши обёртку над ингестом, которая логирует прогресс по файлам.',
    meta_query='ингест файлов без расширения, прогресс-логирование',
)
print(prompt)

## 9. Сравнение трёх моделей на одном запросе

Каждая модель строит свою коллекцию (`pkh_bge-m3`, `pkh_qwen3`, `pkh_e5-instruct`) — поэтому первый прогон по новой модели делает полный ингест. Дальше — только дельта.

**Внимание:** прогон ингеста для всех трёх займёт времени и места на Drive ×3. Если хочешь только сравнить retrieval — сначала прогоняй ячейку ниже на маленькой подвыборке `CHATS_DIR`.

In [ ]:
MODELS = ['bge-m3', 'qwen3', 'e5-instruct']
QUERY = 'как я настраивал ChromaDB на Colab'

snipers = {}
for m in MODELS:
    print(f'=== ingest with {m} ===')
    s = ContextSniper(embedding_model=m)
    print(s.ingest(config.CHATS_DIR))
    snipers[m] = s

for m, s in snipers.items():
    print(f'\n========== {m} ==========')
    res = s.search_context(QUERY, top_k=3)
    for doc, meta, dist in zip(
        res['documents'][0], res['metadatas'][0], res['distances'][0]
    ):
        print(f'[{meta["chat_id"]}] dist={dist:.3f}  {doc[:200]}')